In [1]:
from pathlib import Path
import psycopg
from psycopg.rows import dict_row

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

DB_SETTINGS = {
    "host": "/home/jovyan/.gbr-pg-socket",
    "dbname": "georgia_bid_readiness",
    "user": "jovyan",
    "connect_timeout": 10,
}

In [2]:
views_sql = """
CREATE SCHEMA IF NOT EXISTS reporting;

-- One row per document version.
CREATE OR REPLACE VIEW reporting.document_inventory AS
SELECT
    s.solicitation_id,
    s.agency_name,
    s.solicitation_number,
    s.title,
    d.document_id,
    d.document_type,
    d.amendment_number,
    v.sha256,
    v.page_count,
    v.size_bytes,
    v.local_path,
    observations.observation_count,
    observations.first_retrieved_at,
    observations.last_retrieved_at
FROM core.solicitations AS s
JOIN core.documents AS d
  ON d.solicitation_id = s.solicitation_id
JOIN core.document_versions AS v
  ON v.document_id = d.document_id
LEFT JOIN LATERAL (
    SELECT
        COUNT(*) AS observation_count,
        MIN(o.retrieved_at) AS first_retrieved_at,
        MAX(o.retrieved_at) AS last_retrieved_at
    FROM core.document_observations AS o
    WHERE o.document_id = v.document_id
      AND o.version_sha256 = v.sha256
      AND o.status = 'ok'
) AS observations ON TRUE;

-- One row per extraction run and document version.
CREATE OR REPLACE VIEW reporting.extraction_quality AS
SELECT
    d.solicitation_id,
    p.extraction_run_id,
    p.document_id,
    p.document_sha256,
    COUNT(*) AS page_records,
    COUNT(*) FILTER (
        WHERE p.status = 'ok'
    ) AS successful_pages,
    COUNT(*) FILTER (
        WHERE p.status = 'failed'
    ) AS failed_pages,
    COUNT(*) FILTER (
        WHERE p.needs_review
    ) AS originally_flagged_pages,
    COUNT(*) FILTER (
        WHERE p.needs_review
          AND latest.review_status = 'resolved'
          AND latest.ocr_required = FALSE
    ) AS resolved_flagged_pages,
    COUNT(*) FILTER (
        WHERE p.needs_review
          AND (
              latest.review_status IS DISTINCT FROM 'resolved'
              OR latest.ocr_required IS DISTINCT FROM FALSE
          )
    ) AS unresolved_flagged_pages,
    COALESCE(SUM(p.character_count), 0) AS extracted_characters
FROM core.page_extractions AS p
JOIN core.documents AS d
  ON d.document_id = p.document_id
LEFT JOIN LATERAL (
    SELECT r.review_status, r.ocr_required
    FROM core.page_reviews AS r
    WHERE r.extraction_run_id = p.extraction_run_id
      AND r.document_id = p.document_id
      AND r.page_number = p.page_number
    ORDER BY r.reviewed_at DESC, r.review_id DESC
    LIMIT 1
) AS latest ON TRUE
GROUP BY
    d.solicitation_id,
    p.extraction_run_id,
    p.document_id,
    p.document_sha256;

-- One row per recorded ingestion or extraction run.
CREATE OR REPLACE VIEW reporting.processing_history AS
SELECT
    r.run_id,
    r.run_type,
    r.parent_run_id,
    r.status,
    r.started_at,
    r.completed_at,
    EXTRACT(EPOCH FROM (
        r.completed_at - r.started_at
    )) AS recorded_elapsed_seconds,
    r.artifact_path,
    (
        SELECT COUNT(*)
        FROM core.document_observations AS o
        WHERE o.run_id = r.run_id
    ) AS document_observations,
    (
        SELECT COUNT(*)
        FROM core.page_extractions AS p
        WHERE p.extraction_run_id = r.run_id
    ) AS page_records
FROM core.pipeline_runs AS r;
"""

views_path = PROJECT / "warehouse/models/001_reporting_views.sql"
views_path.parent.mkdir(parents=True, exist_ok=True)

if views_path.exists():
    if views_path.read_text(encoding="utf-8") != views_sql:
        raise RuntimeError(
            "A different reporting SQL file already exists. "
            "Review it before replacing it."
        )
else:
    views_path.write_text(views_sql, encoding="utf-8")

with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:
        cursor.execute(views_sql)

print("Three reporting views created.")
print("SQL saved:", views_path)

Three reporting views created.
SQL saved: /home/jovyan/project/georgia-bid-readiness/warehouse/models/001_reporting_views.sql


In [3]:
SOLICITATION_ID = "commerce-ga-25-012"
EXTRACTION_RUN_ID = "20261016T225302199674Z"

with psycopg.connect(
    **DB_SETTINGS,
    row_factory=dict_row,
) as connection:
    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                COUNT(*) AS document_versions,
                SUM(page_count) AS source_pages,
                SUM(observation_count) AS observations
            FROM reporting.document_inventory
            WHERE solicitation_id = %s;
        """, (SOLICITATION_ID,))
        inventory = dict(cursor.fetchone())

        cursor.execute("""
            SELECT
                SUM(page_records) AS page_records,
                SUM(successful_pages) AS successful_pages,
                SUM(failed_pages) AS failed_pages,
                SUM(originally_flagged_pages) AS flagged_pages,
                SUM(resolved_flagged_pages) AS resolved_pages,
                SUM(unresolved_flagged_pages) AS unresolved_pages
            FROM reporting.extraction_quality
            WHERE solicitation_id = %s
              AND extraction_run_id = %s;
        """, (SOLICITATION_ID, EXTRACTION_RUN_ID))
        quality = dict(cursor.fetchone())

        cursor.execute("""
            SELECT
                run_type,
                status,
                document_observations,
                page_records
            FROM reporting.processing_history
            ORDER BY started_at, run_id;
        """)
        history = cursor.fetchall()

print("DOCUMENT INVENTORY")
for key, value in inventory.items():
    print(f"{key}: {value}")

print("\nEXTRACTION QUALITY")
for key, value in quality.items():
    print(f"{key}: {value}")

print("\nPROCESSING HISTORY")
for row in history:
    print(dict(row))

assert inventory["document_versions"] == 12
assert inventory["source_pages"] == 28
assert inventory["observations"] == 24

assert quality["page_records"] == 28
assert quality["successful_pages"] == 28
assert quality["failed_pages"] == 0
assert quality["flagged_pages"] == 1
assert quality["resolved_pages"] == 1
assert quality["unresolved_pages"] == 0

print("\nReporting totals verified.")

DOCUMENT INVENTORY
document_versions: 12
source_pages: 28
observations: 24

EXTRACTION QUALITY
page_records: 28
successful_pages: 28
failed_pages: 0
flagged_pages: 1
resolved_pages: 1
unresolved_pages: 0

PROCESSING HISTORY
{'run_type': 'ingestion', 'status': 'ok', 'document_observations': 12, 'page_records': 0}
{'run_type': 'ingestion', 'status': 'ok', 'document_observations': 12, 'page_records': 0}
{'run_type': 'extraction', 'status': 'ok', 'document_observations': 0, 'page_records': 28}

Reporting totals verified.


In [4]:
import importlib.metadata as metadata
import os

for package in ["streamlit", "jupyter-server-proxy"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print(
    "Jupyter service prefix:",
    os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "Not available"),
)

streamlit: NOT INSTALLED
jupyter-server-proxy: 4.1.0
Jupyter service prefix: /user/ishittu1@students.kennesaw.edu/
